In [1]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd 

import itertools
from collections import defaultdict

from scripts.ss_graphgen import *

In [2]:
dttz = pd.read_csv("../data/adult.csv")
dtz = dttz.drop_duplicates().reset_index(drop=True)

print("Original number of rows: ", len(dtz))
print(" ")

seln = 500

dtx = dtz.sample(n=seln, replace=False, random_state=42).reset_index(drop=True)
dt = dtx.drop_duplicates().reset_index(drop=True)
dt


Original number of rows:  32537
 


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,31,7,33308,8,11,2,3,0,4,1,0,0,40,39,0
1,24,1,130534,9,13,4,4,1,4,1,0,0,40,39,0
2,60,2,98350,15,10,2,8,0,1,1,0,0,60,30,0
3,31,4,398988,15,10,4,4,1,4,1,0,0,40,39,0
4,43,1,144778,9,13,4,4,1,4,1,0,0,40,39,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,32,4,234755,11,9,5,3,4,2,1,0,0,40,39,0
496,26,4,233777,11,9,2,7,0,4,1,0,0,40,39,1
497,34,6,306982,9,13,2,12,0,1,1,0,0,60,35,0
498,62,4,226733,11,9,2,1,0,4,1,0,0,40,39,1


In [3]:
target_column = "income"

X = dt.drop([target_column], axis=1).values
y = dt[target_column].values
y = np.where(y == 1, 1, -1) 

s = dt["sex"].values

scaler = StandardScaler()
X = scaler.fit_transform(X)



In [4]:
##########
np.random.seed(42)
n = X.shape[0]
indices = np.random.permutation(n)
split = int(0.9 * n)
lhs_idx = indices[:split]
rhs_idx = indices[split:]

lhs_idx_negative = lhs_idx[y[lhs_idx] == -1]
X_LHS_random = X[lhs_idx_negative]
X_RHS_random = X[rhs_idx]
y_RHS_random = y[rhs_idx]

s_LHS_random = s[lhs_idx_negative]
s_RHS_random = s[rhs_idx]

xxneg = y[lhs_idx]

len(lhs_idx_negative), len(lhs_idx), len(rhs_idx), np.sum(xxneg == -1), np.sum(xxneg == 1)


(328, 450, 50, 328, 122)

In [5]:
##########
##########
edges_random, labels_random, graph_random = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                                 XRHS=X_RHS_random, 
                                                                 yRHS=y_RHS_random, 
                                                                 sLHS=s_LHS_random, 
                                                                 sRHS=s_RHS_random,                                                                 
                                                                 k_min=1, 
                                                                 k_max=2)

r_edges_random = graph_random['edges']
r_labels_random = graph_random['labels']

r_graph_structure_random = {
    f"x_{x}": [f"t_{t} ({'+' if r_labels_random[t] == 1 else '-'})" for t in neighbors]
    for x, neighbors in r_edges_random.items()
}

print("Random bipartite graph")
for agent, targets in list(r_graph_structure_random.items())[:10]:
    print(f"{agent}: {targets}")
    
    
print("positive ", list({t for t, l in r_labels_random.items() if l == 1})[:10])
print("negative ", list({t for t, l in r_labels_random.items() if l == -1})[:10])
    
graph_random['n'], graph_random['m']


Random bipartite graph
x_0: ['t_44 (-)', 't_13 (-)']
x_1: ['t_24 (-)', 't_45 (-)']
x_2: ['t_24 (-)', 't_21 (-)']
x_3: ['t_1 (-)', 't_38 (-)']
x_4: ['t_17 (-)', 't_43 (+)']
x_5: ['t_1 (-)', 't_38 (-)']
x_6: ['t_48 (-)', 't_34 (-)']
x_7: ['t_0 (-)', 't_37 (-)']
x_8: ['t_36 (-)', 't_39 (-)']
x_9: ['t_37 (-)', 't_18 (+)']
positive  [3, 5, 6, 8, 9, 10, 11, 40, 43, 46]
negative  [0, 1, 2, 4, 7, 12, 13, 14, 15, 16]


(328, 49)

In [6]:
##########
edges_random, labels_random, graph_random = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                                    XRHS=X_RHS_random, 
                                                                    yRHS=y_RHS_random, 
                                                                    sLHS=s_LHS_random, 
                                                                    sRHS=s_RHS_random,                                                                  
                                                                    threshold=0.2, 
                                                                    topk=False, 
                                                                    thresh=True)


r_edges_random = graph_random['edges']
r_labels_random = graph_random['labels']

r_graph_structure_random = {
    f"x_{x}": [f"t_{t} ({'+' if r_labels_random[t] == 1 else '-'})" for t in neighbors]
    for x, neighbors in r_edges_random.items()
}

print("Random bipartite graph")
for agent, targets in list(r_graph_structure_random.items())[:10]: 
    print(f"{agent}: {targets}")
    
    
print("positive ", list({t for t, l in r_labels_random.items() if l == 1})[:10])
print("negative ", list({t for t, l in r_labels_random.items() if l == -1})[:10])
    
graph_random['n'], graph_random['m']


Random bipartite graph
x_0: []
x_1: []
x_2: []
x_3: []
x_4: []
x_5: []
x_6: []
x_7: []
x_8: []
x_9: []
positive  []
negative  []


(328, 0)

## General

In [7]:
#############
##knn##
#############
graphsRandom1 = []
graphsinfor = []
graphid = 0
for kmax1 in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    _, _, graphx_random1 = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                XRHS=X_RHS_random, 
                                                yRHS=y_RHS_random, 
                                                sLHS=s_LHS_random, 
                                                sRHS=s_RHS_random,                                            
                                                k_min=1, 
                                                k_max=kmax1)
    
    graphx_random1.update({"k_max": kmax1, "knn": "yes", "thresh": "no"})
    graphsRandom1.append(graphx_random1)
    
    ########
    avg_left_deg, avg_left_pos_deg, avg_left_neg_deg, avg_right_deg, avg_overlap, \
    avg_pos_overlap, avg_neg_overlap, only_pos, only_neg, empty_adj, unipos  = getconnectivity_info(graphx_random1["edges"], graphx_random1["labels"])

    graphsinfor.append({
        "Dataset (d)": "Adult ("+ str(X.shape[1]) + ")",
        "kmax": kmax1, 
        "n": graphx_random1['n'],
        "m": graphx_random1['m'],
        "#+ves": len({t for t, l in graphx_random1['labels'].items() if l == 1}),
        "#-ves": len({t for t, l in graphx_random1['labels'].items() if l == -1}), 
        "avg LHSd": avg_left_deg, 
        "avg LHS+d": avg_left_pos_deg, 
        "avg LHS-d": avg_left_neg_deg,  
        "avg RHSd": avg_right_deg,         
        "avg overlap": avg_overlap,
        "avg overlap+": avg_pos_overlap,
        "avg overlap-": avg_neg_overlap,
        "only+Ns": only_pos,
        "only-Ns": only_neg,
        "emptyNs": empty_adj,
        "uni+": unipos,     
        "graphID": graphid
    })
    
    graphid += 1

######
randomgraphsinfo = pd.DataFrame(graphsinfor)   

######
np.save("../graphs/adult_knn_random.npy", np.array(graphsRandom1, dtype=object), allow_pickle=True)
randomgraphsinfo.to_csv("../graphs/adult_knn_graphsummary.npy", index=False)
######
randomgraphsinfo
 


,Dataset (d),kmax,n,m,#+ves,#-ves,avg LHSd,avg LHS+d,avg LHS-d,avg RHSd,avg overlap,avg overlap+,avg overlap-,only+Ns,only-Ns,emptyNs,uni+,graphID
0,Adult (14),1,328,46,10,36,1.0,0.213415,0.786585,7.130435,0.031998,0.006694,0.025304,70,258,0,0,0
1,Adult (14),2,328,49,12,37,2.0,0.484756,1.515244,13.387755,0.111602,0.029089,0.082513,15,184,0,0,1
2,Adult (14),3,328,49,12,37,3.0,0.777439,2.222561,20.081633,0.270381,0.084564,0.185817,4,121,0,0,2
3,Adult (14),4,328,49,12,37,4.0,1.076220,2.923780,26.775510,0.487283,0.173249,0.314034,2,86,0,0,3
4,Adult (14),5,328,49,12,37,5.0,1.405488,3.594512,33.469388,0.764768,0.300608,0.464161,1,61,0,0,4
5,Adult (14),6,328,49,12,37,6.0,1.725610,4.274390,40.163265,1.108693,0.453662,0.655031,0,44,0,0,5
6,Adult (14),7,328,49,12,37,7.0,2.009146,4.990854,46.857143,1.502890,0.601197,0.901693,0,33,0,0,6
7,Adult (14),8,328,49,12,37,8.0,2.347561,5.652439,53.551020,1.963527,0.794119,1.169408,0,19,0,0,7
8,Adult (14),9,328,49,12,37,9.0,2.689024,6.310976,60.244898,2.481875,1.012643,1.469232,0,12,0,0,8
9,Adult (14),10,328,49,12,37,10.0,2.957317,7.042683,66.938776,3.037052,1.209797,1.827254,0,10,0,0,9


In [8]:
#############
##thresh##
#############
graphsRandom11 = []
graphsinfor11 = []
graphid11 = 0
for t11 in [4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10]:
    _, _, graphx_random11 = graphgen_thresh_topk(XLHS=X_LHS_random, 
                                                XRHS=X_RHS_random, 
                                                yRHS=y_RHS_random, 
                                                sLHS=s_LHS_random, 
                                                sRHS=s_RHS_random, 
                                                threshold=t11, 
                                                topk=False, 
                                                thresh=True)
    
    graphx_random11.update({"threshold": t11, "knn": "no", "thresh": "yes"})
    graphsRandom11.append(graphx_random11)

    ########
    avg_left_deg11, avg_left_pos_deg11, avg_left_neg_deg11, avg_right_deg11, avg_overlap11, \
    avg_pos_overlap11, avg_neg_overlap11, only_pos11, only_neg11, empty_adj11, unipos11  = getconnectivity_info(graphx_random11["edges"], graphx_random11["labels"])


    graphsinfor11.append({
        "Dataset (d)": "Adult ("+ str(X.shape[1]) + ")",
        "r": t11, 
        "n": graphx_random11['n'],
        "m": graphx_random11['m'],
        "#+ves": len({t for t, l in graphx_random11['labels'].items() if l == 1}),
        "#-ves": len({t for t, l in graphx_random11['labels'].items() if l == -1}), 
        "avg LHSd": avg_left_deg11, 
        "avg LHS+d": avg_left_pos_deg11, 
        "avg LHS-d": avg_left_neg_deg11,  
        "avg RHSd": avg_right_deg11,         
        "avg overlap": avg_overlap11,
        "avg overlap+": avg_pos_overlap11,
        "avg overlap-": avg_neg_overlap11,  
        "only+Ns": only_pos11,
        "only-Ns": only_neg11,
        "emptyNs": empty_adj11,
        "uni+": unipos11,
        "graphID": graphid11
    })
    
    graphid11 += 1

#######
randomgraphsinfo11 = pd.DataFrame(graphsinfor11)   
    
    
#######   
np.save("../graphs/adult_thresh_random.npy", np.array(graphsRandom11, dtype=object), allow_pickle=True)
randomgraphsinfo11.to_csv("../graphs/adult_thresh_graphsummary.npy", index=False)
#######
randomgraphsinfo11


,Dataset (d),r,n,m,#+ves,#-ves,avg LHSd,avg LHS+d,avg LHS-d,avg RHSd,avg overlap,avg overlap+,avg overlap-,only+Ns,only-Ns,emptyNs,uni+,graphID
0,Adult (14),4.0,328,48,12,36,12.695122,3.719512,8.975610,86.750000,4.561858,1.525352,3.036506,6,51,35,0,0
1,Adult (14),4.5,328,49,12,37,19.545732,5.262195,14.283537,130.836735,9.621176,2.785671,6.835505,2,33,21,0,1
2,Adult (14),5.0,328,49,12,37,27.003049,6.871951,20.131098,180.755102,17.002780,4.458725,12.544055,2,20,9,0,2
3,Adult (14),5.5,328,49,12,37,33.466463,8.344512,25.121951,224.020408,24.895193,6.197862,18.697331,0,9,6,0,3
4,Adult (14),6.0,328,49,12,37,38.951220,9.612805,29.338415,260.734694,32.537136,7.907118,24.630018,0,6,2,0,4
5,Adult (14),6.5,328,49,12,37,42.829268,10.585366,32.243902,286.693878,38.457224,9.440442,29.016782,1,4,0,0,5
6,Adult (14),7.0,328,49,12,37,45.698171,11.286585,34.411585,305.897959,43.116991,10.646770,32.470221,0,2,0,0,6
7,Adult (14),7.5,328,49,12,37,47.350610,11.695122,35.655488,316.959184,45.946297,11.404863,34.541434,0,0,0,0,7
8,Adult (14),8.0,328,49,12,37,48.256098,11.887195,36.368902,323.020408,47.579007,11.776535,35.802473,0,0,0,2,8
9,Adult (14),8.5,328,49,12,37,48.698171,11.960366,36.737805,325.979592,48.411520,11.920955,36.490565,0,0,0,6,9
